## Alexander Rees
## Andrew Lotocki
## Project Phase I
## DS 4420


# Scrape Data From CFB and Kaggle

In [1]:
!{sys.executable} -m pip install --upgrade kagglehub requests
!pip install pandas==2.2.2 numpy==1.26.4 requests==2.32.4 --force-reinstall
!pip install python-dotenv
!pip install kagglehub

zsh:1: parse error near `-m'
  Using cached pandas-2.2.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached charset_normalizer-3.4.5-cp312-cp312-macosx_10_13_universal2.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.2.2-cp312-cp312-macosx_11_0_arm64.whl (11.3 MB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
Using cached requests-2.32.4-py3

In [2]:
import sys
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import re
import requests
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

/Users/arees/DS 4420/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
combine_file = 'nfl_combine_2010_to_2023.csv'
headers = {"Authorization": f"Bearer {api_key}"}

combine_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, 'thomassshaw/nfl-combine-performance-dataset', path=combine_file)



years = list(range(2008, 2024))
cfb_list = []

combine_df["Player"] = combine_df["Player"].str.lower().str.strip()

print("Combine data:")
print(combine_df.head())


def clean_name(name):
    name = name.lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name)
    suffixes = [" jr", " sr", " ii", " iii", " iv", " v"]
    for s in suffixes:
        if name.endswith(s):
            name = name.replace(s, "")
    return name.strip()

combine_df["clean_name"] = combine_df["Player"].apply(clean_name)

for year in years:
    url = f"https://api.collegefootballdata.com/stats/player/season?year={year}"
    r = requests.get(url, headers=headers)
    data = r.json()
    if len(data) == 0:
        continue
    df = pd.DataFrame(data)
    df["player"] = df["player"].str.lower().str.strip()
    df["clean_name"] = df["player"].apply(clean_name)
    df["season"] = year
    cfb_list.append(df)



cfb_df = pd.concat(cfb_list)
print("College stats sample:")
print(cfb_df.head())
cfb_df = pd.merge(cfb_df, combine_df[["clean_name","Year"]], on="clean_name", how="inner")

cfb_df["stat"] = cfb_df["stat"].astype(str)
cfb_df["stat"] = cfb_df["stat"].str.split("-").str[0]
cfb_df["stat"] = pd.to_numeric(cfb_df["stat"], errors="coerce")
cfb_df = cfb_df[cfb_df["stat"] < 10000]
cfb_df = cfb_df[cfb_df["season"] <= cfb_df["Year"]]

last_season = cfb_df.groupby("clean_name")["season"].max().reset_index()
cfb_df = pd.merge(cfb_df, last_season, on=["clean_name","season"], how="inner")

cfb_wide = cfb_df.pivot_table(index=["clean_name","season"], columns=["category","statType"], values="stat", aggfunc="sum").reset_index()
cfb_wide.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in cfb_wide.columns]
cfb_wide.columns.name = None

merged_df = pd.merge( combine_df, cfb_wide, on="clean_name", how="left")
merged_df.to_csv("player_data.csv", index=False)

print(merged_df.head())

Combine data:
   Year             Player Pos         School Height  Weight  40yd  Vertical  \
0  2010     seyi ajirotutu  WR   Fresno State    6-3   204.0  4.60      36.0   
1  2010         rahim alem  DE            LSU    6-3   251.0  4.75      30.5   
2  2010  charles alexander  DT            LSU    6-4   300.0  5.40       NaN   
3  2010  danario alexander  WR       Missouri    6-5   215.0  4.62       NaN   
4  2010         nate allen   S  South Florida    6-0   207.0  4.50       NaN   

   Bench  Broad Jump  3Cone  Shuttle  Drafted  Round  Pick  
0   14.0       115.0   7.22     4.39    False    NaN   NaN  
1    NaN       106.0   7.54     4.80    False    NaN   NaN  
2    NaN         NaN    NaN      NaN    False    NaN   NaN  
3    NaN         NaN    NaN      NaN    False    NaN   NaN  
4   16.0         NaN    NaN      NaN     True    2.0  37.0  
College stats sample:
   season playerId           player position  team     conference  \
0    2008    27200  michael johnson       WR  UN

# Seperating Cleaning and Scaling our Data

In [3]:
df = pd.read_csv("player_data.csv")
df = df.drop(columns=['season', 'Year', 'clean_name', 'Round', 'School'])
df["Height"] = (df["Height"].astype(str).str.split("-").apply(lambda x: int(x[0]) * 12 + int(x[1]) if len(x) == 2 else None))

qb_data = df[df["Pos"] == "QB"]
qb_data = qb_data.dropna(axis=1, thresh=len(qb_data) * 0.5).reset_index(drop=True)

rb_data = df[df["Pos"] == "RB"]
rb_data = rb_data.dropna(axis=1, thresh=len(rb_data) * 0.5).reset_index(drop=True)

cb_data = df[df["Pos"] == "CB"]
cb_data = cb_data.dropna(axis=1, thresh=len(cb_data) * 0.5).reset_index(drop=True)

wr_data = df[df["Pos"] == "WR"]
wr_data = wr_data.dropna(axis=1, thresh=len(wr_data) * 0.5).reset_index(drop=True)

de_data = df[df["Pos"] == "DE"]
de_data = de_data.dropna(axis=1, thresh=len(de_data) * 0.5).reset_index(drop=True)

In [4]:
print(qb_data.shape)
qb_data.head()

(238, 23)


,Player,Pos,Height,Weight,40yd,Vertical,Broad Jump,3Cone,Shuttle,Drafted,...,passing_INT,passing_PCT,passing_TD,passing_YDS,passing_YPA,rushing_CAR,rushing_LONG,rushing_TD,rushing_YDS,rushing_YPC
0,sam bradford,QB,76.0,236.0,4.79,NaN,NaN,NaN,NaN,True,...,0.0,0.565,2.0,562.0,8.1,4.0,5.0,0.0,NaN,NaN
1,jarrett brown,QB,75.0,224.0,4.50,34.5,114.0,7.24,4.39,False,...,9.0,0.632,11.0,2144.0,7.2,118.0,36.0,6.0,452.0,3.8
2,levi brown,QB,75.0,229.0,4.93,31.5,106.0,7.07,4.43,True,...,9.0,0.637,23.0,4254.0,8.4,54.0,37.0,1.0,7.0,0.1
3,sean canfield,QB,76.0,223.0,4.93,29.5,100.0,7.26,4.39,True,...,7.0,0.679,21.0,3271.0,7.3,48.0,7.0,2.0,NaN,NaN
4,daryll clark,QB,74.0,235.0,4.72,NaN,NaN,NaN,NaN,False,...,10.0,0.609,24.0,3003.0,7.9,84.0,51.0,7.0,211.0,2.5


In [5]:
print(rb_data.shape)
rb_data.head()

(426, 22)


,Player,Pos,Height,Weight,40yd,Vertical,Bench,Broad Jump,3Cone,Shuttle,...,receiving_LONG,receiving_REC,receiving_TD,receiving_YDS,receiving_YPR,rushing_CAR,rushing_LONG,rushing_TD,rushing_YDS,rushing_YPC
0,joique bell,RB,71.0,220.0,4.68,36.5,NaN,120.0,6.84,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,jahvid best,RB,70.0,199.0,4.34,32.5,18.0,113.0,6.75,4.17,...,51.0,22.0,4.0,213.0,9.7,141.0,93.0,12.0,867.0,6.1
2,legarrette blount,RB,72.0,241.0,4.70,35.0,18.0,117.0,6.85,4.49,...,7.0,2.0,0.0,13.0,6.5,22.0,30.0,2.0,82.0,3.7
3,chris brown,RB,70.0,210.0,4.55,36.0,17.0,115.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,andre dixon,RB,73.0,205.0,4.56,40.0,NaN,117.0,6.99,4.19,...,27.0,11.0,1.0,112.0,10.2,239.0,45.0,14.0,1093.0,4.6


In [6]:
print(cb_data.shape)
cb_data.head()

(491, 16)


,Player,Pos,Height,Weight,40yd,Vertical,Bench,Broad Jump,3Cone,Shuttle,Drafted,Pick,interceptions_AVG,interceptions_INT,interceptions_TD,interceptions_YDS
0,javier arenas,CB,69.0,197.0,4.60,NaN,NaN,112.0,NaN,NaN,True,50.0,5.0,5.0,0.0,25.0
1,cornelius brown,CB,71.0,198.0,4.59,31.5,11.0,118.0,6.87,4.24,False,NaN,NaN,NaN,NaN,NaN
2,crezdon butler,CB,72.0,191.0,4.43,39.5,17.0,121.0,7.08,4.23,True,164.0,47.0,1.0,0.0,47.0
3,nolan carroll,CB,71.0,204.0,4.42,NaN,NaN,NaN,NaN,NaN,True,145.0,NaN,NaN,NaN,NaN
4,chris chancellor,CB,69.0,177.0,4.49,34.0,14.0,115.0,6.85,4.07,False,NaN,37.0,1.0,0.0,37.0


In [7]:
print(wr_data.shape)
wr_data.head()

(670, 20)


,Player,Pos,Height,Weight,40yd,Vertical,Bench,Broad Jump,3Cone,Shuttle,Drafted,Pick,receiving_LONG,receiving_REC,receiving_TD,receiving_YDS,receiving_YPR,rushing_CAR,rushing_LONG,rushing_TD
0,seyi ajirotutu,WR,75.0,204.0,4.60,36.0,14.0,115.0,7.22,4.39,False,NaN,46.0,49.0,7.0,677.0,13.8,NaN,NaN,NaN
1,danario alexander,WR,77.0,215.0,4.62,NaN,NaN,NaN,NaN,NaN,False,NaN,84.0,113.0,14.0,1781.0,15.8,1.0,10.0,0.0
2,alric arnett,WR,74.0,188.0,4.52,40.0,NaN,122.0,7.03,4.43,False,NaN,46.0,43.0,3.0,586.0,13.6,NaN,NaN,NaN
3,brandon banks,WR,67.0,149.0,4.37,31.0,NaN,113.0,6.88,4.29,False,NaN,64.0,56.0,1.0,705.0,12.6,11.0,20.0,0.0
4,chris bell,WR,74.0,211.0,4.50,35.0,15.0,117.0,6.76,4.35,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print(de_data.shape)
de_data.head()

(261, 12)


,Player,Pos,Height,Weight,40yd,Vertical,Bench,Broad Jump,3Cone,Shuttle,Drafted,Pick
0,rahim alem,DE,75.0,251.0,4.75,30.5,NaN,106.0,7.54,4.80,False,NaN
1,tyson alualu,DE,74.0,295.0,4.87,35.5,21.0,116.0,7.15,4.43,True,10.0
2,kevin basped,DE,76.0,258.0,4.75,29.0,26.0,104.0,7.54,4.88,False,NaN
3,alex carrington,DE,77.0,285.0,4.92,NaN,26.0,NaN,NaN,NaN,True,72.0
4,jermaine cunningham,DE,75.0,266.0,4.89,NaN,NaN,NaN,NaN,NaN,True,53.0


# Building Our MLP Draft Prediction Model

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np

mlp_features = wr_data.drop(columns=["Player", "Pos", "Drafted", "Pick"])
mlp_features = mlp_features.select_dtypes(include=[np.number])
mlp_target = wr_data["Drafted"].astype(int).loc[mlp_features.index]

n_before = len(mlp_features)
mlp_features = mlp_features.dropna()
mlp_target = mlp_target.loc[mlp_features.index]
n_after = len(mlp_features)
print(f"Null drop: {n_before - n_after} rows removed ({n_before} -> {n_after} players)")

pct_drafted = mlp_target.mean() * 100
print(f"Percentage of players drafted (in MLP sample): {pct_drafted:.1f}%")

X_train, X_test, y_train, y_test = train_test_split(
    mlp_features, mlp_target, test_size=0.2, random_state=42, stratify=mlp_target
)

mlp_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(2,), activation="relu", learning_rate_init=0.01, max_iter=400, random_state=12)
)

mlp_model.fit(X_train, y_train)

y_pred = mlp_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("MLP WR drafted vs. not-drafted — Accuracy:", round(acc, 3), "| F1:", round(f1, 3))

Null drop: 516 rows removed (670 -> 154 players)
Percentage of players drafted (in MLP sample): 61.7%
MLP WR drafted vs. not-drafted — Accuracy: 0.742 | F1: 0.8


/Users/arees/DS 4420/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
